# Data Preparation — create the fixed train/val/test splits

One-time preprocessing. Loads the raw MIT-BIH CSVs, splits the train set 80/20, **z-score normalizes each split independently** (per feature, using that split's own mean/std), casts the label (final column) to **int**, and saves ready-to-use arrays to `data/splits/`.

Each saved `.npy` is one array per split: `[187 normalized features | label]` (float32; the label is the last column, a whole number 0–4). The framework then only *loads* these files.

Re-run only when you deliberately want to rebuild the splits.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

RAW = Path(r"C:\Users\dhina\OneDrive\Documents\University\2026\COS 700\Repo\COS700\data\raw")
SPLITS = Path(r"C:\Users\dhina\OneDrive\Documents\University\2026\COS 700\Repo\COS700\data\splits")
SPLITS.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

---
## 0. Raw Data Quality Checks

Read-only inspection of both raw CSVs — **no cleaning, no changes**.
Covers: shape & dtypes · missing values · duplicate rows · class distribution · outliers (IQR method).

In [3]:
# Load raw CSVs once — shared across all checks below
raw_train = pd.read_csv(RAW / "mitbih_train.csv", header=None)
raw_test  = pd.read_csv(RAW / "mitbih_test.csv",  header=None)

print("=== SHAPE ===")
print(f"  train: {raw_train.shape}  (rows × cols)")
print(f"  test:  {raw_test.shape}")

print("\n=== DATA TYPES ===")
dtypes_train = raw_train.dtypes.value_counts()
dtypes_test  = raw_test.dtypes.value_counts()
print("  train dtype counts:", dtypes_train.to_dict())
print("  test  dtype counts:", dtypes_test.to_dict())

# Expected: all float64 for features, float64 for label column (whole numbers stored as float)
unexpected_train = raw_train.dtypes[~raw_train.dtypes.isin([np.float64, np.int64, "float64", "int64"])]
if unexpected_train.empty:
    print("  [OK] No unexpected dtypes in train.")
else:
    print("  [WARN] Unexpected dtypes in train:\n", unexpected_train)

=== SHAPE ===
  train: (87554, 188)  (rows × cols)
  test:  (21892, 188)

=== DATA TYPES ===
  train dtype counts: {dtype('float64'): 188}
  test  dtype counts: {dtype('float64'): 188}
  [WARN] Unexpected dtypes in train:
 0      float64
1      float64
2      float64
3      float64
4      float64
        ...   
183    float64
184    float64
185    float64
186    float64
187    float64
Length: 188, dtype: object


In [4]:
print("=== MISSING VALUES ===")
for name, df in [("train", raw_train), ("test", raw_test)]:
    total_missing = df.isnull().sum().sum()
    cols_with_nan = (df.isnull().sum() > 0).sum()
    print(f"  {name}: {total_missing} missing values across {cols_with_nan} columns")
    if total_missing > 0:
        print(df.isnull().sum()[df.isnull().sum() > 0])

=== MISSING VALUES ===
  train: 0 missing values across 0 columns
  test: 0 missing values across 0 columns


In [5]:
print("=== DUPLICATE ROWS ===")
for name, df in [("train", raw_train), ("test", raw_test)]:
    n_dupes = df.duplicated().sum()
    pct = 100 * n_dupes / len(df)
    print(f"  {name}: {n_dupes} duplicate rows ({pct:.2f}%)")

# Cross-set leakage check — any train rows that also appear in test?
merged = raw_train.merge(raw_test, how="inner")
print(f"\n  train↔test overlap (exact row match): {len(merged)} rows")

=== DUPLICATE ROWS ===
  train: 0 duplicate rows (0.00%)
  test: 0 duplicate rows (0.00%)

  train↔test overlap (exact row match): 0 rows


In [6]:
print("=== CLASS DISTRIBUTION (label column = col 187) ===")
for name, df in [("train", raw_train), ("test", raw_test)]:
    counts = df.iloc[:, -1].value_counts().sort_index()
    pcts   = df.iloc[:, -1].value_counts(normalize=True).sort_index() * 100
    print(f"\n  {name}:")
    for cls in counts.index:
        print(f"    class {int(cls)}: {counts[cls]:6d} rows  ({pcts[cls]:.2f}%)")

# Confirm label values are exactly {0,1,2,3,4} — no stray floats or extra classes
all_labels = pd.concat([raw_train.iloc[:, -1], raw_test.iloc[:, -1]]).unique()
print(f"\n  Unique label values (both sets): {sorted(all_labels)}")
expected = {0.0, 1.0, 2.0, 3.0, 4.0}
if set(all_labels) == expected:
    print("  [OK] Labels are exactly {0,1,2,3,4}.")
else:
    print("  [WARN] Unexpected label values found.")

=== CLASS DISTRIBUTION (label column = col 187) ===

  train:
    class 0:  72471 rows  (82.77%)
    class 1:   2223 rows  (2.54%)
    class 2:   5788 rows  (6.61%)
    class 3:    641 rows  (0.73%)
    class 4:   6431 rows  (7.35%)

  test:
    class 0:  18118 rows  (82.76%)
    class 1:    556 rows  (2.54%)
    class 2:   1448 rows  (6.61%)
    class 3:    162 rows  (0.74%)
    class 4:   1608 rows  (7.35%)

  Unique label values (both sets): [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
  [OK] Labels are exactly {0,1,2,3,4}.


In [7]:
print("=== OUTLIERS — IQR method on feature columns only (col 0–186) ===")
print("  An observation is flagged if ANY feature value falls outside [Q1 - 1.5·IQR, Q3 + 1.5·IQR].\n")

for name, df in [("train", raw_train), ("test", raw_test)]:
    features = df.iloc[:, :-1]          # 187 signal columns
    Q1  = features.quantile(0.25)
    Q3  = features.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    # Per-feature outlier counts
    out_mask = (features < lower) | (features > upper)   # (rows × 187) bool
    outlier_cells = out_mask.sum().sum()
    outlier_rows  = out_mask.any(axis=1).sum()
    features_affected = (out_mask.sum() > 0).sum()

    # Top 10 features by outlier count
    top = out_mask.sum().sort_values(ascending=False).head(10)

    print(f"  {name}:")
    print(f"    rows with ≥1 outlier feature : {outlier_rows:6d} / {len(df)} ({100*outlier_rows/len(df):.1f}%)")
    print(f"    total outlier (cell) count   : {outlier_cells:,}")
    print(f"    features with any outliers   : {features_affected} / {features.shape[1]}")
    print(f"    top 10 features by outlier count (col index → count):")
    for col, cnt in top.items():
        print(f"      col {col:3d}: {cnt}")
    print()

=== OUTLIERS — IQR method on feature columns only (col 0–186) ===
  An observation is flagged if ANY feature value falls outside [Q1 - 1.5·IQR, Q3 + 1.5·IQR].

  train:
    rows with ≥1 outlier feature :  83843 / 87554 (95.8%)
    total outlier (cell) count   : 1,009,139
    features with any outliers   : 186 / 187
    top 10 features by outlier count (col index → count):
      col 126: 21818
      col 127: 20875
      col 128: 20076
      col 129: 19301
      col 125: 19067
      col 130: 18590
      col 131: 17917
      col 124: 17414
      col 132: 17309
      col 133: 16677

  test:
    rows with ≥1 outlier feature :  20963 / 21892 (95.8%)
    total outlier (cell) count   : 250,445
    features with any outliers   : 186 / 187
    top 10 features by outlier count (col index → count):
      col 126: 5472
      col 127: 5255
      col 128: 5028
      col 129: 4820
      col 125: 4744
      col 130: 4647
      col 131: 4477
      col 124: 4370
      col 132: 4299
      col 133: 4130



In [8]:
print("=== FEATURE VALUE RANGE SUMMARY (raw, before normalisation) ===")
for name, df in [("train", raw_train), ("test", raw_test)]:
    features = df.iloc[:, :-1]
    desc = features.describe().T   # one row per feature
    print(f"\n  {name} — features (col 0–186):")
    print(f"    global min  : {desc['min'].min():.6f}")
    print(f"    global max  : {desc['max'].max():.6f}")
    print(f"    mean of means : {desc['mean'].mean():.6f}")
    print(f"    mean of stds  : {desc['std'].mean():.6f}")
    zero_var = (desc['std'] == 0).sum()
    print(f"    zero-variance features: {zero_var}")

=== FEATURE VALUE RANGE SUMMARY (raw, before normalisation) ===

  train — features (col 0–186):
    global min  : 0.000000
    global max  : 1.000000
    mean of means : 0.174283
    mean of stds  : 0.175429
    zero-variance features: 0

  test — features (col 0–186):
    global min  : 0.000000
    global max  : 1.000000
    mean of means : 0.173479
    mean of stds  : 0.174365
    zero-variance features: 0


## 1. Load the train set and split 80/20

In [9]:
train_full = pd.read_csv(RAW / "mitbih_train.csv", header=None)
print("raw train:", train_full.shape)   # (87554, 188) = 187 features + 1 label

train_df, val_df = train_test_split(
    train_full, test_size=0.20, random_state=RANDOM_STATE, shuffle=True,
    stratify=train_full.iloc[:, -1],
)
print("train:", train_df.shape, "| val:", val_df.shape)

# Verify class proportions are preserved
for name, df in [("train", train_df), ("val", val_df)]:
    counts = df.iloc[:, -1].value_counts(normalize=True).sort_index()
    print(f"{name} class proportions:", counts.round(3).to_dict())

raw train: (87554, 188)
train: (70043, 188) | val: (17511, 188)
train class proportions: {0.0: 0.828, 1.0: 0.025, 2.0: 0.066, 3.0: 0.007, 4.0: 0.073}
val class proportions: {0.0: 0.828, 1.0: 0.025, 2.0: 0.066, 3.0: 0.007, 4.0: 0.073}


## 2. Z-score each split independently, int labels, save

Each split is normalized with **its own** per-feature mean/std (independent of the others). The label column is cast to int. Saved as `[features | label]` float32.

In [10]:
def zscore_per_feature(X):
    """Per-feature (per-column) z-score using this array's own statistics."""
    mu = X.mean(axis=0, keepdims=True)
    sd = X.std(axis=0, keepdims=True)
    sd[sd == 0] = 1.0  # guard zero-variance features
    return (X - mu) / sd

def process(df):
    X = df.iloc[:, :-1].to_numpy(dtype=np.float32)   # 187 features
    y = df.iloc[:, -1].to_numpy().astype(np.int64)   # label -> int (0..4)
    X = zscore_per_feature(X).astype(np.float32)
    combined = np.column_stack([X, y]).astype(np.float32)  # [features | label]
    return combined, y

train_arr, y_train = process(train_df)
val_arr,   y_val   = process(val_df)

np.save(SPLITS / "train_idx.npy", train_arr)
np.save(SPLITS / "val_idx.npy",   val_arr)

print("saved train_idx.npy:", train_arr.shape, "| val_idx.npy:", val_arr.shape)
print("train label counts:", dict(zip(*np.unique(y_train, return_counts=True))))
print("val   label counts:", dict(zip(*np.unique(y_val,   return_counts=True))))
print("label column (as int):", np.unique(train_arr[:, -1].astype(int)))

saved train_idx.npy: (70043, 188) | val_idx.npy: (17511, 188)
train label counts: {np.int64(0): np.int64(57977), np.int64(1): np.int64(1778), np.int64(2): np.int64(4630), np.int64(3): np.int64(513), np.int64(4): np.int64(5145)}
val   label counts: {np.int64(0): np.int64(14494), np.int64(1): np.int64(445), np.int64(2): np.int64(1158), np.int64(3): np.int64(128), np.int64(4): np.int64(1286)}
label column (as int): [0 1 2 3 4]


## 3. Same for the test set

The official test set is not split — normalize it independently (its own stats), int labels, save to `test_idx.npy`.

In [11]:
test_df = pd.read_csv(RAW / "mitbih_test.csv", header=None)
print("raw test:", test_df.shape)

test_arr, y_test = process(test_df)
np.save(SPLITS / "test_idx.npy", test_arr)

print("saved test_idx.npy:", test_arr.shape)
print("test label counts:", dict(zip(*np.unique(y_test, return_counts=True))))

raw test: (21892, 188)
saved test_idx.npy: (21892, 188)
test label counts: {np.int64(0): np.int64(18118), np.int64(1): np.int64(556), np.int64(2): np.int64(1448), np.int64(3): np.int64(162), np.int64(4): np.int64(1608)}


## 4. Sanity check — reload the saved files

In [12]:
for name in ("train_idx", "val_idx", "test_idx"):
    a = np.load(SPLITS / f"{name}.npy")
    X, y = a[:, :-1], a[:, -1].astype(int)
    print(f"{name}: array {a.shape} | X {X.shape} float32 | y {y.shape} ints {np.unique(y)}")

train_idx: array (70043, 188) | X (70043, 187) float32 | y (70043,) ints [0 1 2 3 4]
val_idx: array (17511, 188) | X (17511, 187) float32 | y (17511,) ints [0 1 2 3 4]
test_idx: array (21892, 188) | X (21892, 187) float32 | y (21892,) ints [0 1 2 3 4]
